# שבוע 4: היכרות עם Google Colab וציוני דרך

בשיעור זה נלמד:
- כיצד לנווט בסביבת Colab
- כיצד לטעון קואורדינטות ציוני דרך מקבצי TPS
- כיצד לצייר ציוני דרך
- מה אנחנו מחפשים בנתוני צורה

> **הוראות**: הריצו כל תא בסדר מלמעלה למטה. לחצו על התא ואחר כך `Shift+Enter`.

In [ ]:
# התקנת חבילות והגדרת סביבת העבודה
!pip install python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

## חלק 1: מה זה קובץ TPS?

פורמט TPS הוא הפורמט הסטנדרטי לנתוני ציוני דרך בריפומטריה גאומטרית.
כל פרט מתואר כך:

```
LM=8
123.4 456.7
...
ID=specimen_name
```

- **LM=8**: 8 ציוני דרך
- **שורות הקואורדינטות**: x y לכל ציון דרך
- **ID**: שם הפרט

In [ ]:
import urllib.request

# טעינת נתוני TPS של מטבעות הדריאנוס מ-GitHub
url = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/hadrian.tps'
try:
    with urllib.request.urlopen(url) as response:
        tps_text = response.read().decode('utf-8')
    print('שורות ראשונות של הקובץ:')
    print('\n'.join(tps_text.split('\n')[:20]))
except Exception as e:
    print(f'שגיאה בטעינה: {e}')
    print('יוצרים נתוני דוגמה...')
    # נתוני דוגמה למקרה של בעיית רשת
    tps_text = ''
    np.random.seed(42)
    for i in range(20):
        tps_text += 'LM=8\n'
        cx, cy = np.random.uniform(200,400), np.random.uniform(200,400)
        for j in range(8):
            angle = 2*np.pi*j/8
            r = 80 + np.random.randn()*5
            tps_text += f'{cx+r*np.cos(angle):.2f} {cy+r*np.sin(angle):.2f}\n'
        tps_text += f'ID=hadrian_{i+1:03d}\n'
    print('נוצרו נתוני דוגמה')

## חלק 2: פירוש קובץ TPS

In [ ]:
def parse_tps(text):
    """פירוש קובץ TPS לרשימת מערכי numpy"""
    specimens = []
    ids = []
    lines = text.strip().split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                x, y = float(parts[0]), float(parts[1])
                coords.append([x, y])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1])
        i += 1
    return np.array(specimens), ids

landmarks, ids = parse_tps(tps_text)
print(f'נטענו {len(landmarks)} פרטים')
print(f'כל פרט: {landmarks.shape[1]} ציוני דרך x {landmarks.shape[2]} קואורדינטות')
print(f'שמות ראשונים: {ids[:5]}')

## חלק 3: ציור ציוני הדרך

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle(rtl('ציוני דרך — מטבעות הדריאנוס (6 דוגמאות)'), fontsize=14)

for idx, ax in enumerate(axes.flat):
    if idx < len(landmarks):
        lm = landmarks[idx]
        ax.scatter(lm[:, 0], lm[:, 1], c='steelblue', s=60, zorder=3)
        # חיבור ציוני הדרך בסדר
        ax.plot(np.append(lm[:, 0], lm[0, 0]),
                np.append(lm[:, 1], lm[0, 1]), 'gray', alpha=0.4, linewidth=1)
        for j, (x, y) in enumerate(lm):
            ax.annotate(str(j+1), (x, y), textcoords='offset points',
                       xytext=(4, 4), fontsize=8)
        ax.set_aspect('equal')
        label = ids[idx] if idx < len(ids) else f'פרט {idx+1}'
        ax.set_title(label, fontsize=9)
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## חלק 4: גיוון הצורה

בואו נשכב את כל ציוני הדרך על גרף אחד — האם כל המטבעות נראים דומים?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for i, lm in enumerate(landmarks):
    ax.plot(lm[:, 0], lm[:, 1], 'o-', alpha=0.3, markersize=4,
            color=plt.cm.viridis(i/len(landmarks)))

ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_title(rtl(f'כל {len(landmarks)} פרטים — ציוני דרך גולמיים'), fontsize=13)
ax.set_xlabel(rtl('קואורדינטת X (פיקסלים)'))
ax.set_ylabel(rtl('קואורדינטת Y (פיקסלים)'))
plt.tight_layout()
plt.show()

print(rtl('שימו לב: הצורות אינן מיושרות — כל מטבע צולם מזווית שונה'))

## תרגיל

1. כמה ציוני דרך יש לכל מטבע? באיזה חלק של המטבע ממוקם ציון דרך מספר 1?
2. מדוע הצורות הגולמיות אינן מיושרות? מה צריך לעשות בשביל להשוות אותן?
3. שנו את `idx` בתא הציור כדי לראות מטבעות שונים — האם אתם מבחינים בהבדלים?